In [ ]:
df = pd.read_csv('/content/all_data_with_rank_and_point.csv')
df["home_possession"] = df["home_possession"].str.replace('%', '').astype(float)
df["away_possession"] = df["away_possession"].str.replace('%', '').astype(float)

zero_cols = ['home_red_cards', 'away_red_cards', 'home_yellow_cards', 'away_yellow_cards',
             'home_goalkeeper_saves', 'away_goalkeeper_saves', 'home_offsides', 'away_offsides',
             'home_attempted_passes', 'home_successful_passes', 'away_attempted_passes', 'away_successful_passes']
df[zero_cols] = df[zero_cols].fillna(0)
df.dropna(subset=['home_rank', 'away_rank', 'home_points', 'away_points'], inplace=True)
df["result"] = np.sign(df["home_goals"] - df["away_goals"])

for col in ['home_team', 'away_team', 'league', 'season']:
    df[col + "_enc"] = LabelEncoder().fit_transform(df[col])

df['game_week'] = df['game_week'].replace({'Relegation Round': 39, 'Finals': 40, 'Relegation Decider': 41})
df['game_week'] = df['game_week'].astype(str).str.extract(r'(\d+)').astype(float).fillna(0).astype(int)
df = df.sort_values(by=["season", "game_week"]).reset_index(drop=True)

# 📊 حساب recent form و average stats


In [ ]:

def get_recent_stats(df, team_id, idx, n=5):
    past = df.iloc[:idx]
    matches = past[(past["home_team_enc"] == team_id) | (past["away_team_enc"] == team_id)].tail(n)
    wins = draws = losses = gs = gc = 0
    for _, r in matches.iterrows():
        if r["home_team_enc"] == team_id:
            gs += r["home_goals"]; gc += r["away_goals"]
            if r["result"] == 1: wins += 1
            elif r["result"] == 0: draws += 1
            else: losses += 1
        elif r["away_team_enc"] == team_id:
            gs += r["away_goals"]; gc += r["home_goals"]
            if r["result"] == -1: wins += 1
            elif r["result"] == 0: draws += 1
            else: losses += 1
    total = len(matches)
    return wins/total if total else 0, draws/total if total else 0, losses/total if total else 0, gs/total if total else 0, gc/total if total else 0

def get_avg_stat(df, team_id, col_home, col_away, idx, n=5):
    past = df.iloc[:idx]
    matches = past[(past["home_team_enc"] == team_id) | (past["away_team_enc"] == team_id)].tail(n)
    values = []
    for _, r in matches.iterrows():
        if r["home_team_enc"] == team_id:
            values.append(r[col_home])
        elif r["away_team_enc"] == team_id:
            values.append(r[col_away])
    return np.mean(values) if values else 0

# استخراج الميزات


In [ ]:
home_recent_wins, home_draws, home_losses = [], [], []
away_recent_wins, away_draws, away_losses = [], [], []
home_avg_goals_scored, home_avg_goals_conceded = [], []
away_avg_goals_scored, away_avg_goals_conceded = [], []
home_avg_shots, home_avg_corners, home_avg_possession, home_avg_pass_acc = [], [], [], []
away_avg_shots, away_avg_corners, away_avg_possession, away_avg_pass_acc = [], [], [], []
for idx, row in df.iterrows():
    h, a = row["home_team_enc"], row["away_team_enc"]
    h_stats = get_recent_stats(df, h, idx)
    a_stats = get_recent_stats(df, a, idx)
    home_recent_wins.append(h_stats[0]); home_draws.append(h_stats[1]); home_losses.append(h_stats[2])
    home_avg_goals_scored.append(h_stats[3]); home_avg_goals_conceded.append(h_stats[4])
    away_recent_wins.append(a_stats[0]); away_draws.append(a_stats[1]); away_losses.append(a_stats[2])
    away_avg_goals_scored.append(a_stats[3]); away_avg_goals_conceded.append(a_stats[4])
    home_avg_shots.append(get_avg_stat(df, h, "home_shots", "away_shots", idx))
    home_avg_corners.append(get_avg_stat(df, h, "home_corners", "away_corners", idx))
    home_avg_possession.append(get_avg_stat(df, h, "home_possession", "away_possession", idx))
    home_avg_pass_acc.append(get_avg_stat(df, h, "home_successful_passes", "away_successful_passes", idx))
    away_avg_shots.append(get_avg_stat(df, a, "home_shots", "away_shots", idx))
    away_avg_corners.append(get_avg_stat(df, a, "home_corners", "away_corners", idx))
    away_avg_possession.append(get_avg_stat(df, a, "home_possession", "away_possession", idx))
    away_avg_pass_acc.append(get_avg_stat(df, a, "home_successful_passes", "away_successful_passes", idx))

df["home_recent_wins"] = home_recent_wins
df["home_recent_draws"] = home_draws
df["home_recent_losses"] = home_losses
df["home_avg_goals_scored"] = home_avg_goals_scored
df["home_avg_goals_conceded"] = home_avg_goals_conceded
df["away_recent_wins"] = away_recent_wins
df["away_recent_draws"] = away_draws
df["away_recent_losses"] = away_losses
df["away_avg_goals_scored"] = away_avg_goals_scored
df["away_avg_goals_conceded"] = away_avg_goals_conceded
df["home_avg_shots"] = home_avg_shots
df["home_avg_corners"] = home_avg_corners
df["home_avg_possession"] = home_avg_possession
df["home_avg_pass_acc"] = home_avg_pass_acc
df["away_avg_shots"] = away_avg_shots
df["away_avg_corners"] = away_avg_corners
df["away_avg_possession"] = away_avg_possession
df["away_avg_pass_acc"] = away_avg_pass_acc


# ميزات إضافية


In [ ]:
df['point_diff'] = df['home_points'] - df['away_points']
df['rank_diff'] = df['away_rank'] - df['home_rank']
df['form_diff'] = df['home_recent_wins'] - df['away_recent_wins']

# المواجهات المباشرة


In [ ]:
h2h_home_win_rate, h2h_draw_rate, h2h_loss_rate = [], [], []
for idx, row in df.iterrows():
    h, a = row["home_team_enc"], row["away_team_enc"]
    past = df.iloc[:idx]
    h2h = past[((past['home_team_enc'] == h) & (past['away_team_enc'] == a)) | ((past['home_team_enc'] == a) & (past['away_team_enc'] == h))]
    if len(h2h) == 0:
        h2h_home_win_rate.append(0); h2h_draw_rate.append(0); h2h_loss_rate.append(0)
        continue
    wins = sum(((r['home_team_enc'] == h and r['result'] == 1) or (r['away_team_enc'] == h and r['result'] == -1)) for _, r in h2h.iterrows())
    draws = sum(r['result'] == 0 for _, r in h2h.iterrows())
    losses = len(h2h) - wins - draws
    total = len(h2h)
    h2h_home_win_rate.append(wins / total)
    h2h_draw_rate.append(draws / total)
    h2h_loss_rate.append(losses / total)

df['h2h_home_win_rate'] = h2h_home_win_rate
df['h2h_home_draw_rate'] = h2h_draw_rate
df['h2h_home_loss_rate'] = h2h_loss_rate

# 🔁 نموذج Oversampling


In [ ]:
wins = df[df['result'] == 1]
losses = df[df['result'] == -1]
draws = df[df['result'] == 0]
draws_upsampled = resample(draws, replace=True, n_samples=max(len(wins), len(losses)), random_state=42)
df = pd.concat([wins, losses, draws_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)


# ⚖️ Standard Scaling


In [ ]:

columns_to_scale = [
    'home_recent_wins', 'home_recent_draws', 'home_recent_losses',
    'home_avg_goals_scored', 'home_avg_goals_conceded',
    'away_recent_wins', 'away_recent_draws', 'away_recent_losses',
    'away_avg_goals_scored', 'away_avg_goals_conceded',
    'home_avg_shots', 'home_avg_corners', 'home_avg_possession', 'home_avg_pass_acc',
    'away_avg_shots', 'away_avg_corners', 'away_avg_possession', 'away_avg_pass_acc',
    'point_diff', 'rank_diff', 'form_diff',
    'h2h_home_win_rate', 'h2h_home_draw_rate', 'h2h_home_loss_rate'
]

target = 'result'
features = [
    'game_week', 'home_rank', 'home_points', 'away_rank', 'away_points',
    'season_enc',"home_recent_draws", 'league_enc',
    'home_team_enc', 'away_team_enc',"home_recent_losses",'away_recent_draws', 'away_recent_losses',
    'home_recent_wins', 'away_recent_wins', 'form_diff',
    'home_avg_goals_scored', 'home_avg_goals_conceded',
    'away_avg_goals_scored', 'away_avg_goals_conceded',
    'home_avg_shots', 'home_avg_corners', 'home_avg_possession', 'home_avg_pass_acc',
    'away_avg_shots', 'away_avg_corners', 'away_avg_possession', 'away_avg_pass_acc',
    'point_diff', 'rank_diff',
    'h2h_home_win_rate', 'h2h_home_draw_rate', 'h2h_home_loss_rate'
]

y = df[target]
new_x=df[features]

# StandardScaler

In [ ]:
scaler = StandardScaler()
new_x = new_x.copy()
new_x[columns_to_scale] = scaler.fit_transform(new_x[columns_to_scale])
new_x=pd.concat([new_x,y],axis=1)
new_x

In [ ]:
df_train, df_test = train_test_split(new_x, test_size=0.2, shuffle=True, random_state=42)

In [ ]:
df_train = df_train.loc[:, ~df_train.columns.duplicated()]
print(df_train.columns)

In [ ]:
df_test = df_test.loc[:, ~df_test.columns.duplicated()]

print(df_test.columns)

performance = predictor.evaluate(df_test)
print(performance)

In [ ]:
predictor_over = TabularPredictor(label=target, eval_metric='accuracy')
predictor_over.fit(train_data=df_train)

In [ ]:
performance = predictor_over.evaluate(df_test)
performance